# Refinitiv / LSEG Data — Example Workflows

Covers:
1. Session setup
2. Identifier mapping (ISIN → RIC, SEDOL, ticker → RIC)
3. Snapshot fundamentals — single period
4. Multi-period fundamentals panel
5. Price time series — single stock
6. Price time series — multi-stock, returns
7. Estimates (IBES/StarMine)
8. ESG scores
9. Full universe pull via constituent DataFrame

In [ ]:
import pandas as pd
import numpy as np
import lseg.data as ld

from quant_tools.refinitiv import RefinitivClient

---
## 1 · Session setup

Config lives at `~/.lseg-data/lseg-data.config.json`.  
The `RefinitivClient` context manager opens/closes the session automatically.

In [ ]:
# Verify connectivity — prints session info
ld.open_session("platform")
print(ld.session.Definition().get_session())
ld.close_session()

---
## 2 · Identifier mapping

### 2a — ISIN → RIC

In [ ]:
with RefinitivClient() as rl:
    isins = [
        "US0378331005",  # Apple
        "US5949181045",  # Microsoft
        "GB0007980591",  # BP
        "NL0010273215",  # ASML
        "JP3633400001",  # Toyota
        "DK0062498333",  # Novo Nordisk
    ]

    df = ld.get_data(
        universe=[f"ISIN:{i}" for i in isins],
        fields=["TR.RIC", "TR.CommonName", "TR.ExchangeName", "TR.GICSSector"],
    )
    df.columns = ["isin_ric", "ric", "name", "exchange", "sector"]
    print(df.to_string())

### 2b — SEDOL → RIC

In [ ]:
with RefinitivClient() as rl:
    sedols = ["2588173", "0798059", "B09CBL4"]  # HSBC, BP, Vodafone

    df = ld.get_data(
        universe=[f"SEDOL:{s}" for s in sedols],
        fields=["TR.RIC", "TR.CommonName", "TR.ExchangeName"],
    )
    print(df)

### 2c — Bloomberg ticker → RIC

In [ ]:
with RefinitivClient() as rl:
    # Bloomberg format: {ticker} {exchange} Equity  →  Refinitiv: use ticker search
    bbg_tickers = ["AAPL US", "BP/ LN", "AIR FP"]  # US, London, Paris

    df = ld.get_data(
        universe=[f"BBG:{t}" for t in bbg_tickers],
        fields=["TR.RIC", "TR.CommonName"],
    )
    print(df)

### 2d — Add RIC column to a constituent DataFrame

If you already have a quant-tools constituent DataFrame (from iShares), map via ISIN.

In [ ]:
from quant_tools.constituents import fetch

constituents = fetch("ACWI", enrich=True)
print(constituents.columns.tolist())
constituents.head(3)

In [ ]:
# iShares gives us ISIN — batch-map to RIC
def map_isins_to_rics(isins: list[str], batch_size: int = 200) -> pd.Series:
    """Return Series: isin → ric."""
    rows = []
    for i in range(0, len(isins), batch_size):
        batch = isins[i:i+batch_size]
        df = ld.get_data(
            universe=[f"ISIN:{x}" for x in batch],
            fields=["TR.RIC"],
        )
        df.columns = ["query", "ric"]
        df["isin"] = [x.replace("ISIN:", "") for x in df["query"]]
        rows.append(df[["isin", "ric"]])
    result = pd.concat(rows).set_index("isin")["ric"]
    return result

with RefinitivClient() as rl:
    # Use first 50 as demo
    sample = constituents[constituents["isin"].notna()].head(50)
    ric_map = map_isins_to_rics(sample["isin"].tolist())
    constituents_with_ric = constituents.join(ric_map.rename("ric"), on="isin")

print(f"RIC match rate: {constituents_with_ric['ric'].notna().mean():.0%}")
constituents_with_ric[["ticker", "name", "isin", "ric"]].head(10)

---
## 3 · Snapshot fundamentals — single period

In [ ]:
RICS = ["AAPL.O", "MSFT.O", "GOOGL.O", "AMZN.O", "META.O",
        "BP.L",   "SHEL.L", "AIRP.PA", "ASML.AS", "NOVN.S"]

with RefinitivClient() as rl:
    fund = rl.fetch_fundamentals(RICS, period="FY0")

# Key valuation ratios
fund[["market_cap", "revenue", "ebitda", "net_income",
      "pe_trailing", "ev_ebitda", "roe", "net_margin"]].round(2)

---
## 4 · Multi-period fundamentals panel

Pull 5 years of annual fundamentals and build a panel.

In [ ]:
with RefinitivClient() as rl:
    periods = {}
    for yr in range(0, 6):  # FY0 = most recent, FY-5 = 5 years ago
        label = f"FY{-yr}" if yr > 0 else "FY0"
        df = rl.fetch_fundamentals(
            RICS,
            period=label,
            fields={"revenue": "TR.Revenue", "net_income": "TR.NetIncome",
                    "ebitda": "TR.EBITDA", "roe": "TR.ReturnOnEquity",
                    "period_end": "TR.Revenue.periodenddate"},
        )
        df["period"] = label
        periods[label] = df

panel = pd.concat(periods.values()).reset_index()
panel = panel.rename(columns={"index": "ric"})  # adjust if already named
panel = panel.sort_values(["ric", "period"])
print(panel.shape)
panel.head(12)

In [ ]:
# Revenue CAGR (FY-5 → FY0)
rev = panel.pivot(index="period", columns="ric", values="revenue")
cagr = (rev.loc["FY0"] / rev.loc["FY-5"]) ** (1/5) - 1
cagr.sort_values(ascending=False).rename("Revenue CAGR 5yr").to_frame()

---
## 5 · Price time series — single stock

In [ ]:
with RefinitivClient() as rl:
    # Daily OHLCV for Apple, 3 years
    aapl = ld.get_history(
        universe="AAPL.O",
        fields=["TR.PriceClose", "TR.PriceOpen", "TR.PriceHigh",
                "TR.PriceLow", "TR.Volume"],
        interval="1D",
        start="2022-01-01",
    )

aapl.columns = ["close", "open", "high", "low", "volume"]
aapl.index = pd.to_datetime(aapl.index)

print(f"{len(aapl):,} trading days")
aapl.tail()

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(aapl.index, aapl["close"], lw=1.5, color="#3b82f6")
ax1.set_ylabel("Price (USD)")
ax1.set_title("Apple (AAPL.O)")
ax2.bar(aapl.index, aapl["volume"] / 1e6, color="#6b7280", width=1)
ax2.set_ylabel("Volume (M)")
plt.tight_layout()
plt.show()

---
## 6 · Price time series — multi-stock, returns

Monthly closes, USD total returns proxy.

In [ ]:
with RefinitivClient() as rl:
    prices = rl.fetch_prices(
        RICS,
        start="2015-01-01",
        fields=["TR.PriceClose"],
        freq="monthly",
    )

# Flatten MultiIndex columns if needed (field, ric) → ric
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(1)

prices = prices.dropna(how="all", axis=1)
print(prices.shape)
prices.tail(3)

In [ ]:
# Monthly returns
returns = prices.pct_change().dropna(how="all")

# Cumulative total return indexed to 100
cum = (1 + returns).cumprod() * 100

fig, ax = plt.subplots(figsize=(12, 5))
for col in cum.columns:
    ax.plot(cum.index, cum[col], lw=1.3, label=col)
ax.axhline(100, color="white", lw=0.5, alpha=0.3)
ax.set_title("Cumulative return (indexed 100, monthly, local currency)")
ax.legend(fontsize=8, ncol=5)
plt.tight_layout()
plt.show()

In [ ]:
# Annualised stats
ann_ret  = returns.mean() * 12
ann_vol  = returns.std()  * np.sqrt(12)
sharpe   = ann_ret / ann_vol
max_dd   = (cum / cum.cummax() - 1).min()

stats = pd.DataFrame({
    "Ann Return": ann_ret,
    "Ann Vol":    ann_vol,
    "Sharpe":     sharpe,
    "Max DD":     max_dd,
}).sort_values("Sharpe", ascending=False).round(3)

stats

---
## 7 · Estimates (IBES / StarMine)

In [ ]:
with RefinitivClient() as rl:
    est = rl.fetch_estimates(RICS)

est[["eps_est_fy1", "eps_est_fy2", "eps_smart",
     "rev_est_fy1", "ebitda_est_fy1",
     "num_analysts", "rec_mean", "target_price",
     "pe_forward"]].round(2)

In [ ]:
# SmartEstimate vs consensus — divergence flags analyst revision momentum
with RefinitivClient() as rl:
    raw = ld.get_data(
        universe=RICS,
        fields=["TR.EPSMeanEstimate", "TR.EPSSmartEstimate",
                "TR.EPSSmartEstimate.smartestimatedirection"],
        parameters={"Period": "FY1"},
    )

raw.columns = ["ric", "eps_consensus", "eps_smart", "smart_direction"]
raw["smart_vs_consensus"] = (raw["eps_smart"] - raw["eps_consensus"]) / raw["eps_consensus"].abs()
raw.set_index("ric").round(3)

---
## 8 · ESG scores

In [ ]:
with RefinitivClient() as rl:
    esg = rl.fetch_esg(RICS)

esg.sort_values("esg_score", ascending=False)

In [ ]:
# ESG vs financial performance scatter
with RefinitivClient() as rl:
    fund2 = rl.fetch_fundamentals(RICS, fields={
        "roe": "TR.ReturnOnEquity",
        "net_margin": "TR.NetProfitMarginActValue",
    })

combined = esg[["esg_score", "env_score", "social_score", "gov_score"]].join(fund2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes,
    ["env_score", "social_score", "gov_score"],
    ["Environment", "Social", "Governance"]):
    ax.scatter(combined[col], combined["roe"], s=60)
    for ric, row in combined.iterrows():
        ax.annotate(ric.split(".")[0], (row[col], row["roe"]), fontsize=7)
    ax.set_xlabel(f"{label} Score")
    ax.set_ylabel("ROE")
    ax.set_title(f"ROE vs {label}")
plt.tight_layout()
plt.show()

---
## 9 · Full universe pull via constituent DataFrame

Pulls everything for ACWI and saves to `data/`.

In [ ]:
from quant_tools.constituents import fetch

# Load constituents and map ISINs to RICs
acwi = fetch("ACWI", enrich=True)

with RefinitivClient() as rl:
    ric_map = map_isins_to_rics(acwi["isin"].dropna().unique().tolist())

acwi["ric"] = acwi["isin"].map(ric_map)
matched = acwi["ric"].notna().sum()
print(f"RIC coverage: {matched}/{len(acwi)} ({matched/len(acwi):.0%})")

In [ ]:
# Full data pull — runs in ~5 min for 2,000+ stocks
rics_universe = acwi["ric"].dropna().unique().tolist()

with RefinitivClient() as rl:
    data = rl.fetch_all(
        rics_universe,
        output_dir="../data",
        prices=True,
        fundamentals=True,
        estimates=True,
        esg=True,
    )

for k, df in data.items():
    print(f"{k:15s}: {df.shape}")

In [ ]:
# Join everything into a single cross-sectional DataFrame
snapshot = (
    data["fundamentals"]
    .join(data["estimates"][["eps_est_fy1", "pe_forward", "num_analysts", "rec_mean", "target_price"]])
    .join(data["esg"][["esg_score", "env_score", "social_score", "gov_score"]])
)

# Attach sector/country from constituents
meta = acwi.set_index("ric")[["sector", "country", "weight_pct"]]
snapshot = snapshot.join(meta)

print(f"Universe: {len(snapshot):,} stocks × {snapshot.shape[1]:,} columns")
print(f"Null rate: {snapshot.isna().mean().mean():.0%}")
snapshot.head(5)